In [1]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [2]:
from sccnasim.xlib.xvcf import vcf_load
from sccnasim.xlib.xrange import format_chrom

In [3]:
phased_snp_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/base/data/snp/HCC3N.phased.vcf.gz"

seed_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/cellsnp_mode1a/seed.csp_mode1a.h5ad"
seed_xf_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/xf_tag/cellsnp_mode1a/seed.xf_tag.csp_mode1a.h5ad"
rs_xf_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/cellsnp_mode1a/sccnasim.xf_tag.csp_mode1a.h5ad"
scrs_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/cellsnp_mode1a/sccnasim-cs_screadsim.csp_mode1a.h5ad"
scrs_xf_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/align/cellranger/xf_tag/cellsnp_mode1a/sccnasim-cs_screadsim.xf_tag.csp_mode1a.h5ad"

In [4]:
out_fn = "cellsnp_mode1a.stat.tsv"

In [5]:
adata = ad.read_h5ad(seed_fn)
adata

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 600 × 8866
    obs: 'cell', 'cell_type'
    var: 'CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'chrom', 'start', 'end', 'snp', 'hap0', 'hap1'
    layers: 'AD', 'DP', 'OTH'

In [6]:
tmp_fn = out_fn + ".tmp"
fp = open(tmp_fn, "w")
fp.write("\t".join([
    "group", "n_cell", "n_phase", 
    "n_DP1", "n_both_alleles_DP1",
    "n_DP5", "n_both_alleles_DP5",
    "n_DP10", "n_both_alleles_DP10",
    "n_DP20", "n_both_alleles_DP20",
]) + "\n")

for call_fn, label in zip(
    [seed_fn, seed_xf_fn, rs_xf_fn, scrs_fn, scrs_xf_fn],
    ["seed", "seed_xf", "rs_xf", "scrs", "scrs_xf"]
):
    print("processing '%s' ..." % label)
    df_phase, header_phase = vcf_load(phased_snp_fn)

    adata = ad.read_h5ad(call_fn)
    print(adata.shape)

    a = adata.layers['AD'].toarray().sum(axis = 0)
    d = adata.layers['DP'].toarray().sum(axis = 0)
    b = d - a

    fp.write("\t".join([
            label,
            str(adata.shape[0]),
            str(df_phase.shape[0]),
            str(np.sum(d >= 1)),
            str(np.sum(np.logical_and(a >= 1, b >= 1))),
            str(np.sum(d >= 5)),
            str(np.sum(np.logical_and(d >= 5, np.logical_and(a >= 1, b >= 1)))),
            str(np.sum(d >= 10)),
            str(np.sum(np.logical_and(d >= 10, np.logical_and(a >= 1, b >= 1)))),
            str(np.sum(d >= 20)),
            str(np.sum(np.logical_and(d >= 20, np.logical_and(a >= 1, b >= 1))))
        ]) + "\n"
    )

fp.close()

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


processing 'seed' ...
(600, 8866)
processing 'seed_xf' ...
(600, 8866)
processing 'rs_xf' ...


/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


(1200, 8866)
processing 'scrs' ...
(1200, 8866)
processing 'scrs_xf' ...
(1200, 8866)


/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [7]:
df = pd.read_csv(tmp_fn, sep = '\t')
df

,group,n_cell,n_phase,n_DP1,n_both_alleles_DP1,n_DP5,n_both_alleles_DP5,n_DP10,n_both_alleles_DP10,n_DP20,n_both_alleles_DP20
0,seed,600,8866,8308,5770,3855,3715,2171,2164,1207,1207
1,seed_xf,600,8866,4151,3171,2367,2316,1587,1584,984,984
2,rs_xf,1200,8866,4064,2943,3152,2762,2289,2195,1549,1538
3,scrs,1200,8866,8312,547,6395,537,4310,502,2537,451
4,scrs_xf,1200,8866,4068,377,3268,371,2464,364,1701,338


In [8]:
df.index = df['group']
df.index.name = None
del df['group']
df = df.transpose()
df

,seed,seed_xf,rs_xf,scrs,scrs_xf
n_cell,600,600,1200,1200,1200
n_phase,8866,8866,8866,8866,8866
n_DP1,8308,4151,4064,8312,4068
n_both_alleles_DP1,5770,3171,2943,547,377
n_DP5,3855,2367,3152,6395,3268
n_both_alleles_DP5,3715,2316,2762,537,371
n_DP10,2171,1587,2289,4310,2464
n_both_alleles_DP10,2164,1584,2195,502,364
n_DP20,1207,984,1549,2537,1701
n_both_alleles_DP20,1207,984,1538,451,338


In [9]:
df.to_csv(out_fn, sep = '\t', header = True, index = True)